# Lesson 67 · Edit distance from scratch — measuring "how much was misheard": the mathematical foundation of WER

**Learning goals**
1. Understand the three edit-distance operations: insertion, deletion, substitution
2. Implement the O(m×n) dynamic-programming algorithm from scratch
3. Use it to implement the core WER (word error rate) calculation
4. See why ASR evaluation can't do without it

> **An algorithm lesson**: fill in the DP table by hand for small strings, then write the code.

← **Previous lesson** · [L66 · ASR end to end](L66_asr_overview.ipynb)

> Last lesson covered **ASR end to end**: acoustic model → language model → decoder, end-to-end vs classical pipelines.
> This lesson explores **edit distance from scratch**.

## This lesson's story: how does a search box know how many typos you made?

You type "teh cat sat" into the search box, and Google instantly suggests "the cat sat". Behind that is the same dynamic program that "lines two strings up and sees how far apart they are".

**Edit distance**: what is the minimum number of operations needed to turn one string into another? There are only three operations:

| Operation | Example | Cost |
|---|---|---|
| Substitution (S) | `cat` → `cut` (a→u) | 1 |
| Insertion (I) | `cat` → `cats` (add s) | 1 |
| Deletion (D) | `cats` → `cat` (remove s) | 1 |

Note: the edit distance from `teh` to `the` is **2** (two substitutions, e→h and h→e) — standard Levenshtein has no "swap neighbors" operation.

**Why doesn't standard Levenshtein allow swaps?** "Swap e and h" looks like 1 operation, cheaper than 2 substitutions, but historically Levenshtein's definition chose just these three: delete, insert, substitute. Add swaps and you get the **Damerau–Levenshtein distance** — a different algorithm, used in spell checking; but the standard evaluation for speech recognition (WER) has always used the original, so we follow the original here too. Remember: when two strings are aligned, a difference at one position can only be "corrected" by a substitution; there are no "cross-position swaps".

**Key insight**: Levenshtein's dynamic program breaks "aligning two strings" into **subproblems**:

```
dp[i][j] = minimum number of operations to turn a[:i] into b[:j]

if a[i-1] == b[j-1]:  dp[i][j] = dp[i-1][j-1]        # same character, no operation needed
else:                  dp[i][j] = 1 + min(
                           dp[i-1][j],    # delete a[i-1]
                           dp[i][j-1],    # insert b[j-1]
                           dp[i-1][j-1]   # substitute
                       )
```

**Aurora connection**: the WER evaluation tool in `aurora.speech` is exactly the word-level version of this algorithm, used to measure the output quality of an ASR system.

This section's task: first implement the character-level `edit_distance(a, b)`, then build it into the word-level `wer(reference, hypothesis)`.

In [ ]:
import numpy as np

## 💭 Why is this DP guaranteed to be optimal?

When you see `dp[i][j] = 1 + min(three neighbors)`, you might ask: **if we just pick the best subproblem like this, how can we be sure the whole thing is optimal?**

**The intuitive reason (no formal proof needed)**:

The edit-distance problem has **optimal substructure** — the shortest edit path between any two strings can be broken down into **the shortest edit path between shorter strings** + **one final operation**.

Take turning `'kitten'` into `'sitting'`:
- The shortest path must look like this: first turn `'kitte'` into some prefix of `'sittin'` (with the fewest operations), then deal with the last characters n and g
- Or: first turn a prefix of `'kitten'` into `'sittin'`, then append or delete at the end

At each step we **only need to know which of the three neighbors is cheapest** — because the last operation into `dp[i][j]` has to come from one of those three cells, and each of them already holds the best possible cost for its own prefixes. This isn't a greedy guess: the `min()` checks every possible last step, and **optimal substructure** guarantees that's enough — which is the heart of dynamic programming.

**A more rigorous argument** (optional): suppose some path doesn't take the best neighbor in the `min()` but a worse one; its cost on reaching `dp[i][j]` is already larger. Its total cost at the end, `dp[m][n]`, must then be larger too (since there's no way to "optimize backwards"), so this path can't be the global optimum. By contradiction: only the `min()` choices can piece together the globally optimal path.

Remember: **every cell of the DP table is "the optimal cost from the start up to here"**; once the whole table is filled in order, the bottom-right cell `dp[m][n]` is the overall global optimum.

## 🔬 Fill one cell before you code: how the DP table actually grows

`dp[i][j] = 1 + min(three neighbors)` above is abstract and hard to pin down the first time. Fill in one cell yourself with the smallest example, `a="ab"` → `b="ac"`, and the formula becomes concrete right away.

**First, understand the coordinates**:
- What `dp[i][j]` means: **the minimum number of operations to turn the first `i` characters of `a` into the first `j` characters of `b`**
- So `dp[0][j]` is going from the **empty string** to the first `j` characters of `b` (only possible with `j` **insertions**)
- `dp[i][0]` is going from the first `i` characters of `a` to the **empty string** (only possible with `i` **deletions**)
- That's why `dp[i][0] = i` and `dp[0][j] = j`

So we need an **(m+1) × (n+1)** table (not m×n), because row 0 and column 0 are the **boundary conditions** — they stand for conversions to and from the empty string.

`dp[i][j]` = the minimum number of operations to turn the first `i` characters of `a` into the first `j` characters of `b`:

```
        ""   a    c
   ""    0    1    2      ← row 0, dp[0][j]=j (the empty string grows j characters by insertion)
   a     1    0    1
   b     2    1  ┌─?─┐    ← dp[2][2] to fill
                 └───┘
```

**When filling `dp[2][2]`**, we're dealing with `a[1]='b'` and `b[1]='c'`, which are **different**. There are three neighbors to consider:

| Neighbor | Meaning | Formula | Value |
|---|---|---|---|
| `dp[1][1]` (top-left) | first 1 of a and first 1 of b already handled; now **substitute** a[1] ('b') with b[1] ('c') | `dp[1][1] + 1` | `0+1 = 1` |
| `dp[2][1]` (left) | all 2 of a and the first 1 of b already handled; now **insert** b[1] ('c') at the end of a | `dp[2][1] + 1` | `1+1 = 2` |
| `dp[1][2]` (above) | first 1 of a and all 2 of b already handled; now **delete** a[1] ('b') | `dp[1][2] + 1` | `1+1 = 2` |

Take the minimum → `dp[2][2] = 1` (one substitution does it).

**The core pattern** (remember this "gesture"):
- **Same character** (`a[i-1] == b[j-1]`): copy the top-left `dp[i-1][j-1]` directly, no +1 (no operation needed)
- **Different characters** (`a[i-1] != b[j-1]`): look at the three neighbors, add 1, take the minimum
  - top-left `dp[i-1][j-1] + 1` → **substitute** a[i-1] with b[j-1]
  - left `dp[i][j-1] + 1` → **insert** b[j-1] after the current position in a
  - above `dp[i-1][j] + 1` → **delete** a[i-1]

Why do the three neighbors map to those meanings? **An easy way to remember**:
- **Left** (`i` stays, `j` grows): we handle more of b, which means **inserting** b's character after the current position in a
- **Above** (`i` grows, `j` stays): we handle more of a while b's progress stays put, which means **deleting** the extra character of a
- **Top-left** (`i` grows, `j` grows): both sides move forward, which means the characters at these two positions are lined up by a **substitution**

## 💡 A key question: why copy the top-left cell when the characters match?

Someone will ask: **if `a[i-1] == b[j-1]`, why must we choose `dp[i-1][j-1]` without considering the other neighbors?**

**Answer**: because "do nothing" is strictly better than every alternative.

Imagine turning `'cat'` into `'cat'`:
- Choose "do nothing": `0` steps
- Choose "delete then insert" (an alternative): delete 'c', then insert 'c' = `2` steps ⟹ certainly worse than doing nothing
- Any other detour costs extra

Mathematically, when `a[i-1] == b[j-1]`,
```
dp[i-1][j-1]                                       ✓ optimal
1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])    ✗ necessarily ≥ 1 + dp[i-1][j-1]
```

So copying the top-left cell is **not only allowed but required** — any other choice adds a cost for nothing. This is **optimal substructure** in action.

## ✏️ Task 1: implement character-level edit distance

**Spec**:

```python
def edit_distance(a, b) -> int:
    # initialize a (len(a)+1) × (len(b)+1) dp table
    # dp[i][0] = i (delete the first i characters of a)
    # dp[0][j] = j (insert the first j characters of b)
    # fill the table row by row, return dp[m][n]
```

**Acceptance criteria**:

| Input | Expected output | Why |
|---|---|---|
| `('', '')` | `0` | empty → empty, 0 steps |
| `('abc', '')` | `3` | 3 deletions |
| `('abc', 'abc')` | `0` | identical |
| `('kitten', 'sitting')` | `3` | the classic example |
| `('', 'abc')` | `3` | 3 insertions |

**Note**: the function accepts a `str` or a `list` (the later `wer()` will pass in word lists).

### Worked derivation: `"kitten"` → `"sitting"`

This is the classic edit-distance example, and the answer is 3. To help you check your reasoning, here are **the first 3 rows of the DP table** (for the first 3 characters of `a`: k, i, t):

```
        ""   s    i    t    t    i    n    g
   ""    0    1    2    3    4    5    6    7
   k     1    1    2    3    4    5    6    7
   i     2    2    1    2    3    4    5    6
   t     3    3    2    1    2    3    4    5
```

**Row 1 by hand** (a = "k", column by column):
- `dp[1][0] = 1` (initialization: delete k)
- `dp[1][1]`: `a[0]='k'` vs `b[0]='s'` differ → `1 + min(dp[0][1]=1, dp[1][0]=1, dp[0][0]=0) = 1` (substitution)
- `dp[1][2]`: `a[0]='k'` vs `b[1]='i'` differ → `1 + min(dp[0][2]=2, dp[1][1]=1, dp[0][1]=1) = 2`
- …and so on

**Row 2 by hand** (a = "ki"):
- `dp[2][0] = 2` (initialization: delete k and i)
- `dp[2][1]`: `a[1]='i'` vs `b[0]='s'` differ → `1 + min(dp[1][1]=1, dp[2][0]=2, dp[1][0]=1) = 2`
- `dp[2][2]`: `a[1]='i'` vs `b[1]='i'` **match** → `dp[1][1] = 1` (copy the top-left)
- `dp[2][3]`: `a[1]='i'` vs `b[2]='t'` differ → `1 + min(dp[1][3]=3, dp[2][2]=1, dp[1][2]=2) = 2`
- …and so on

**Key observations**:
- When `a[i-1] == b[j-1]` (like the 'i' at row 2, column 2), **copy the top-left** directly, at no extra cost
- Every cell depends on its three neighbors — top-left, above and left — which is the subproblem decomposition at work

The full 7×8 table ends with `dp[6][7] = 3` (all 6 characters of "kitten" → all 7 characters of "sitting").

In [ ]:
def edit_distance(a, b) -> int:
    """Character-level Levenshtein edit distance, pure-Python dynamic programming.

    It also accepts word lists (any indexable sequence), so word-level WER can reuse this function.
    This design (relying only on len() and indexing) lets the same DP code handle sequences of
    characters, words, or any objects at all — it separates the algorithm from the data
    representation, which is the right abstraction.

    Hints:
      dp[i][j] = minimum number of edits to turn a[:i] into b[:j]
               (i.e. the first i elements of a vs the first j elements of b)
      Borders: dp[i][0] = i (delete all), dp[0][j] = j (insert all)
      Recurrence: a[i-1]==b[j-1] → dp[i-1][j-1] (no operation)
             otherwise → 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
                      (minimum of delete, insert, substitute)
    """
    # TODO: fill the DP table using the recurrence above and return dp[m][n]
    raise NotImplementedError("Your implementation: follow the DP recurrence above")

<details>
<summary>📖 <b>Reference answer (SOLUTION)</b> — expand only after finishing Task 1 (this is a Markdown cell: it won't run and won't overwrite your implementation above)</summary>

```python
def edit_distance(a, b) -> int:
    """Character-level Levenshtein edit distance, pure-Python dynamic programming.
    It also accepts word lists (any indexable sequence).
    """
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n]
```

</details>


In [ ]:
# Check
try:
    assert edit_distance('', '') == 0
    assert edit_distance('abc', '') == 3
    assert edit_distance('', 'abc') == 3
    assert edit_distance('kitten', 'sitting') == 3   # the classic example
    assert edit_distance('hello', 'hello') == 0
    print('✅ character-level edit distance checks pass')
except (NotImplementedError, TypeError):
    print('⚠️  edit_distance not implemented yet — please finish Task 1')


## ✏️ Task 2: implement word-level WER

**Signature**:
```python
def wer(reference: str, hypothesis: str) -> float:
    # reference/hypothesis are space-separated strings
    # first .split(), then call edit_distance(word list, word list)
    # return edit_distance / len(ref_words)
    # empty reference: return 0.0 if hypothesis is also empty, else float('inf')
```

**Key design**: reuse `edit_distance()` directly, just swapping "characters" for "words".

**Acceptance criteria**:

| reference | hypothesis | WER |
|---|---|---|
| `'hello world'` | `'hello world'` | `0.0` |
| `'hello world'` | `'hello'` | `0.5` |
| `'a b c'` | `'a x c'` | `0.333...` |
| `''` | `''` | `0.0` |
| `''` | `'anything'` | `inf` (no reference words, nothing to normalize by) |

**Why can WER be > 1.0?**

If a speech recognizer outputs far more words than the reference has (say, through repetition or hallucination), the edit distance gets large. A concrete example:
- **reference** = `'hello'` (1 word)
- **hypothesis** = `'i am very sorry for this mistake'` (7 words)
- **edit distance** = 7: none of the 7 output words matches "hello", so the cheapest fix is 1 substitution (one output word → "hello") + 6 insertions to account for the remaining output words
- **WER** = 7 / 1 = **7.0** (far beyond 100%)

This is rare in practice (real systems are seldom this badly wrong), but the definition allows it; at the other end, a WER near 0% means the system rarely makes mistakes.

**Note**: `len(reference.split())` is N (the number of reference words).

In [ ]:
def wer(reference: str, hypothesis: str) -> float:
    """Word Error Rate = word-level edit distance / number of reference words.

    Hints:
      1. lowercase reference / hypothesis and .split() them into word lists
      2. handle the edge case of empty ref_words (see the code already provided)
      3. call edit_distance(ref_words, hyp_words) to reuse the DP —
         edit_distance works on any indexable sequence, no need to rewrite the DP loop!
      4. return distance / len(ref_words)
    """
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else float('inf')
    # TODO: call edit_distance(ref_words, hyp_words) to compute the word-level distance
    raise NotImplementedError("Your implementation: call edit_distance(ref_words, hyp_words)")


<details>
<summary>📖 <b>Reference answer (SOLUTION)</b> — expand only after finishing Task 2 (this is a Markdown cell: it won't run and won't overwrite your implementation above)</summary>

```python
# Key design: edit_distance accepts any indexable sequence, so word lists work too;
# reuse it directly instead of rewriting the DP loop — that's the right abstraction.

def wer(reference: str, hypothesis: str) -> float:
    """Word Error Rate = word-level edit distance / number of reference words."""
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else float('inf')
    # reuse edit_distance: character level and word level share one DP skeleton; only the input sequence type changes
    return edit_distance(ref_words, hyp_words) / len(ref_words)
```

</details>


In [ ]:
# Check
try:
    assert wer('hello world', 'hello world') == 0.0
    assert wer('hello world', 'hello') == 0.5        # 1 deletion / 2 words
    assert abs(wer('a b c', 'a b c d') - 1/3) < 1e-9 # 1 insertion / 3 ref words

    ref = 'the cat sat on the mat'
    hyp = 'the cat set on mat'
    print(f'ref: {ref}')
    print(f'hyp: {hyp}')
    print(f'WER = {wer(ref, hyp):.2%}')  # sat->set (S), deleted 'the' (D) = 2/6
    print('✅ WER checks pass')
except (NotImplementedError, TypeError):
    print('⚠️  wer not implemented yet — please finish Task 2')


## Alignment backtrace: seeing every error

Backtracking through the dp table tells you whether each error is an S (substitution), I (insertion) or D (deletion).

**About multiple optimal paths**: when several neighbors in the DP table give the `min()` value, there are **several alignment paths with the same edit distance**. The code checks them in order with `if...elif...elif...else` and picks **the first matching operation** (priority: M > S > I > D), so the output is one optimal path, not all of them. This has **no effect** on the WER — WER only cares about the number, the minimum edit distance, not the particular alignment; but when a person inspects the errors, different alignment paths can suggest different linguistic explanations. If you need to enumerate every optimal path, you can change the backtrace into a recursion over all equal-cost neighbors, but usually you don't need to.

In [ ]:
def alignment(ref_str, hyp_str):
    """Print the word-level alignment path, labeled M/S/I/D."""
    a, b = ref_str.split(), hyp_str.split()
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if a[i-1] == b[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    ops, i, j = [], m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and a[i-1] == b[j-1]:
            ops.append(('M', a[i-1], b[j-1])); i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == 1 + dp[i-1][j-1]:
            ops.append(('S', a[i-1], b[j-1])); i -= 1; j -= 1
        elif j > 0 and dp[i][j] == 1 + dp[i][j-1]:
            ops.append(('I', '-', b[j-1])); j -= 1
        else:
            ops.append(('D', a[i-1], '-')); i -= 1
    print(f'{"Op":2s}  {"REF":12s}  {"HYP":12s}')
    print('-' * 30)
    for op, r, h in reversed(ops):
        marker = '  ' if op == 'M' else '**'
        print(f'{marker}{op:2s}  {r:12s}  {h:12s}{marker}')

alignment('the cat sat on the mat', 'the cat set on mat')

## Wrap-up

| Concept | What to remember |
|---|---|
| Edit distance | 3 operations, a DP table of size (m+1)×(n+1) |
| WER | = word-level edit distance / number of reference words; can be > 1.0 |
| Backtrace | walk from dp[m][n] towards the top-left to recover the alignment path |
| L68 | CTC alignment — the same "all alignment paths" idea, but over continuous frames |

The next lesson (L68) replaces hand labeling with CTC alignment: `ctc_alignment` automatically aligns a sequence of acoustic frames with a target word sequence, serving as the training objective for the CTC family of models (such as wav2vec 2.0 and DeepSpeech).

L69's CTC forward algorithm uses the same "fill in a table by recurrence" idea, except that minimizing the edit cost becomes summing log probabilities.

## Aurora connection

The two functions in this lesson correspond to the ASR evaluation tools in the production module `src/aurora/speech/`:

| Function in this lesson | Production path | Notes |
|---|---|---|
| `edit_distance(a, b)` | `src/aurora/speech/metrics.py::edit_distance` | generalized version, accepts any sequence |
| `wer(reference, hypothesis)` | `src/aurora/speech/metrics.py::wer` | delegates to `edit_distance`, no duplicated DP |

Extra features of the production implementation:
- **O(n) space optimization**: keep only the two adjacent rows `dp[i-1]` and `dp[i]`;
  when you only need the distance (not the backtrace path), memory drops from O(m×n) to O(n).
- **Batch WER**: accepts several sentences as a `List[str]` and reports the error rate over the whole corpus.

```python
# Take a look at the production module
from aurora.speech.metrics import edit_distance, wer
print(wer('the cat sat on the mat', 'the cat set on mat'))  # 0.333...
```


## ✏️ Closed-book derivation check cell — the edit-distance DP table

**Rule: close every cell above and do the following derivation from memory alone.**

**Problem**: the Levenshtein distance from "cat" to "cut"

1. Write down the recurrence (three operations: insertion, deletion, substitution)
2. Fill in the complete DP table by hand (4×4, including the border row/column)
3. Backtrack through the table and write down one shortest edit path

|   |   | c | u | t |
|---|---|---|---|---|
|   | 0 | 1 | 2 | 3 |
| c | 1 |   |   |   |
| a | 2 |   |   |   |
| t | 3 |   |   |   |

(Fill in the table and write the path here...)

In [ ]:
# Check: the table you filled in matches the algorithm
import sys; sys.path.insert(0, 'src')

def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if a[i-1] == b[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n], dp

dist, table = levenshtein("cat", "cut")
assert dist == 1, f"expected distance 1, got {dist}"
print("DP table (rows = cat, columns = cut):")
header = "    " + "  ".join([" "] + list("cut"))
print(header)
for i, row_label in enumerate(" cat"):
    print(f"  {row_label} " + "  ".join(str(table[i][j]) for j in range(4)))
print(f"\n✅ Edit distance('cat','cut') = {dist}")

In [ ]:
# ✏️ Self-review for this lesson
l67_review = {
    "levenshtein_dp_memorized":  None,  # can write the DP transition from memory (match → unchanged, mismatch → 1+min(del/ins/sub))? True/False
    "edit_distance_impl":        None,  # edit_distance() implemented correctly, all 5 assertions pass? True/False
    "wer_impl":                  None,  # wer() implemented correctly, word-level version reusing edit_distance? True/False
    "dp_table_traced":           None,  # traced the full 4×4 DP table for "cat"→"cut" (with border row/column) on the whiteboard? True/False
    "whiteboard_passed":         None,  # cleared the "cat"→"cut" whiteboard derivation cell? True/False
}

unfilled = [k for k, v in l67_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l67_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L67 fully cleared! On to L68: how CTC alignment works')

---

→ **Next lesson** · [L68 · How CTC alignment works](L68_ctc_alignment.ipynb)

> Next lesson covers **how CTC alignment works**: the blank symbol, monotonic paths and label collapsing.